# Traffic Demand Prediction: Hackathon Ensemble Submission
**Evaluation Metric:** $\text{Score} = \max(0, 100 \times R^2(\text{actual}, \text{predicted}))$

### Strategy Overview:
* **Validation Strategy:** Robust 5-Fold Cross-Validation to guarantee local validation tracks the leaderboard.
* **Feature Engineering:** Geospatial decoding via Geohashes, cyclical time representations, structural interaction terms, and targeted historical aggregation.
* **The "Secret Sauce" Lag:** The critical pattern relies on temporal consistency. We built a custom dictionary mapper to feed exactly $Day_{48}$ demand as a direct predictive lag feature for $Day_{49}$.
* **Modeling:** Optimized Ensemble blending across three gradient-boosting giants: LightGBM, XGBoost, and CatBoost, with weights mathematically optimized using Nelder-Mead optimization on Out-of-Fold (OOF) predictions.

In [1]:
%pip install numpy pandas pygeohash scikit-learn scipy lightgbm xgboost catboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

## 1. Data Loading & Inspection
Let's load the source files and do a quick sanity check on dataset dimensions and time boundaries to ensure alignment between historical context and the forecast window.

In [3]:
print("Loading train, test, and sample submission files...")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

# Quick shape check
print(f"-> Train set layout: {train.shape[0]:,} rows, {train.shape[1]} columns")
print(f"-> Test set layout:  {test.shape[0]:,} rows, {test.shape[1]} columns")

# Verify day sequences 
train_days = sorted(train['day'].unique())
test_days = sorted(test['day'].unique())
print(f"-> Historical timeline spans days: {train_days[0]} to {train_days[-1]}")
print(f"-> Forecast target spans days:      {test_days}")

Loading train, test, and sample submission files...
-> Train set layout: 77,299 rows, 11 columns
-> Test set layout:  41,778 rows, 10 columns
-> Historical timeline spans days: 48 to 49
-> Forecast target spans days:      [np.int64(49)]


## 2. Core Helper Functions
Creating fast utility functions to parse textual strings into mathematical objects that tree algorithms can cleanly partition on.

In [4]:
def parse_timestamp(ts):
    """
    Translates 'H:MM' string timestamps into absolute minutes from midnight.
    This continuous numerical scale allows trees to easily calculate distance over time.
    """
    h, m = ts.split(":")
    return int(h) * 60 + int(m)

def decode_geohash(gh):
    """
    Converts string geohashes back to raw geographical coordinates (Lat/Lon).
    Wrapped in a try-except block to gracefully handle any noisy or truncated string anomalies.
    """
    try:
        loc = pgh.decode(gh)
        return loc.latitude, loc.longitude
    except Exception:
        return np.nan, np.nan

## 3. Feature Engineering Pipeline
This is our core feature store pipeline. It encapsulates domain insights regarding traffic patterns: diurnal cycles, geographic clustering, structural bottlenecks, and weather interactions.

In [5]:
def engineer_features(df, lag_map=None, geohash_stats=None):
    """
    Applies the full feature extraction pipeline uniformly to train and test splits.
    """
    df = df.copy()

    # --- 1. Temporal & Cyclical Attributes ---
    df["ts_minutes"]  = df["timestamp"].apply(parse_timestamp)
    df["hour"]        = df["ts_minutes"] // 60
    df["minute"]      = df["ts_minutes"] % 60
    
    # Logical binary indicators for expected rush/off-peak conditions
    df["is_peak_am"]  = df["hour"].between(7, 9).astype(int)
    df["is_peak_pm"]  = df["hour"].between(17, 19).astype(int)
    df["is_night"]    = (df["hour"] < 6).astype(int)
    df["is_midnight"] = (df["hour"] == 0).astype(int)
    
    # Cyclical mapping: mapping 23:45 close to 00:00 using trigonometry
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["min_sin"]  = np.sin(2 * np.pi * df["ts_minutes"] / (24 * 60))
    df["min_cos"]  = np.cos(2 * np.pi * df["ts_minutes"] / (24 * 60))

    # --- 2. Geospatial Mapping ---
    geo_decoded = df["geohash"].apply(decode_geohash)
    df["lat"] = geo_decoded.apply(lambda x: x[0])
    df["lon"] = geo_decoded.apply(lambda x: x[1])

    # Prefix slices act as hierarchical spatial groupings (macro vs micro neighborhoods)
    df["geohash_len"]     = df["geohash"].str.len()
    df["geohash_prefix3"] = df["geohash"].str[:3]
    df["geohash_prefix4"] = df["geohash"].str[:4]
    df["geohash_enc"]     = df["geohash"].astype("category").cat.codes

    # --- 3. The Day-48 Direct Match Lag ---
    if lag_map is not None:
        # Match current spatio-temporal slice with exactly day 48 behavior
        df["lag_demand_d48"] = df.apply(
            lambda r: lag_map.get((r["geohash"], r["ts_minutes"]), np.nan), axis=1
        )
        # Robust backfilling strategy for any unmapped spatial grids
        df["lag_demand_d48"] = df["lag_demand_d48"].fillna(df.groupby("geohash")["lag_demand_d48"].transform("mean"))
        df["lag_demand_d48"] = df["lag_demand_d48"].fillna(df["lag_demand_d48"].mean())

    # --- 4. Long-Term Historical Location Baselines ---
    if geohash_stats is not None:
        df = df.merge(geohash_stats, on="geohash", how="left")
        for col in ["gh_demand_mean", "gh_demand_std", "gh_demand_max", "gh_demand_median"]:
            df[col] = df[col].fillna(df[col].mean())

    # --- 5. Categorical Mappings ---
    df["RoadType"] = df["RoadType"].fillna("Unknown")
    df["Weather"]  = df["Weather"].fillna("Unknown")
    
    # Ordered labels to give models a cleaner linear signal if desired
    road_map = {"Highway": 3, "Street": 2, "Residential": 1, "Unknown": 0}
    weather_map = {"Sunny": 4, "Cloudy": 3, "Rainy": 2, "Foggy": 1, "Snowy": 0, "Unknown": -1}
    
    df["RoadType_enc"] = df["RoadType"].map(road_map).fillna(0).astype(int)
    df["Weather_enc"]  = df["Weather"].map(weather_map).fillna(-1).astype(int)
    df["LargeVehicles_enc"] = (df["LargeVehicles"] == "Allowed").astype(int)
    df["Landmarks_enc"]     = (df["Landmarks"] == "Yes").astype(int)

    # --- 6. Environmental and Interaction Term Crosses ---
    df["Temperature"] = df["Temperature"].fillna(df["Temperature"].median())
    df["temp_sq"]     = df["Temperature"] ** 2
    
    # For instance: Rush hour traffic scales directly based on infrastructural capacity (lanes)
    df["peak_x_lanes"] = (df["is_peak_am"] + df["is_peak_pm"]) * df["NumberofLanes"]
    df["temp_x_weather"] = df["Temperature"] * df["Weather_enc"]
    df["lat_x_lon"] = df["lat"] * df["lon"]

    return df

## 4. Contextual Baseline and Global Feature Preparation
Before feeding the arrays into the models, we compile the spatial lookups and isolate historical Day-48 as our blueprint proxy matrix for Day-49 predictions.

In [6]:
print("Isolating Day-48 spatial grids to compute exact target lag map...")
day48 = train[train["day"] == 48].copy()
day48["ts_minutes"] = day48["timestamp"].apply(parse_timestamp)

# Fast O(1) memory lookup for matching space-time demands
lag_map = {(row["geohash"], row["ts_minutes"]): row["demand"] for _, row in day48.iterrows()}
print(f"-> Constructed lag dictionary containing {len(lag_map):,} structural coordinates.")

# Compute continuous global aggregations per location
print("Calculating long-term descriptive stats across location clusters...")
geohash_stats = (
    train.groupby("geohash")["demand"]
    .agg(gh_demand_mean="mean", gh_demand_std="std",
         gh_demand_max="max", gh_demand_median="median")
    .reset_index()
)
geohash_stats["gh_demand_std"] = geohash_stats["gh_demand_std"].fillna(0)

# Apply global transform to both frames
print("\nExecuting comprehensive feature generation...")
train_fe = engineer_features(train, lag_map=lag_map, geohash_stats=geohash_stats)
test_fe  = engineer_features(test,  lag_map=lag_map, geohash_stats=geohash_stats)

Isolating Day-48 spatial grids to compute exact target lag map...
-> Constructed lag dictionary containing 69,427 structural coordinates.
Calculating long-term descriptive stats across location clusters...

Executing comprehensive feature generation...


## 5. Column Verification and Matrix Conversion
Ensure structural alignment between training matrices and inference boundaries.

In [7]:
FEATURE_COLS = [
    "day", "ts_minutes", "hour", "minute",
    "is_peak_am", "is_peak_pm", "is_night", "is_midnight",
    "hour_sin", "hour_cos", "min_sin", "min_cos",
    "lat", "lon", "lat_x_lon",
    "geohash_enc",
    "NumberofLanes",
    "RoadType_enc", "Weather_enc",
    "LargeVehicles_enc", "Landmarks_enc",
    "Temperature", "temp_sq", "temp_x_weather",
    "peak_x_lanes",
    "lag_demand_d48",
    "gh_demand_mean", "gh_demand_std", "gh_demand_max", "gh_demand_median",
]

# Guarantee feature existence across both transforms
FEATURE_COLS = [c for c in FEATURE_COLS if c in train_fe.columns]
print(f"Total features entering the model pipeline: {len(FEATURE_COLS)}")

# Convert to raw numpy structures for faster model iteration
X = train_fe[FEATURE_COLS].values
y = train_fe["demand"].values
X_test = test_fe[FEATURE_COLS].values

print(f"Final Data Shapes -> Train X: {X.shape} | Train y: {y.shape} | Test X: {X_test.shape}")

Total features entering the model pipeline: 30
Final Data Shapes -> Train X: (77299, 30) | Train y: (77299,) | Test X: (41778, 30)


## 6. 5-Fold Cross-Validation Framework
Setting up standard out-of-fold array containers to save model predictions. This lets us reliably evaluate generalization performance locally without risking target leakage.

In [8]:
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Set up arrays to hold out-of-fold validation scores
lgb_oof, xgb_oof, cat_oof = np.zeros(len(X)), np.zeros(len(X)), np.zeros(len(X))
lgb_test_preds, xgb_test_preds, cat_test_preds = np.zeros(len(X_test)), np.zeros(len(X_test)), np.zeros(len(X_test))

## 7. Model Training Round 1: LightGBM
LightGBM handles high-cardinality splits efficiently. We are initializing deep leaf growth tracking here (`num_leaves=256`) to capture higher-order feature combinations.

In [9]:
print("Training LightGBM Regressor across Folds...")

lgb_params = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    0.03,
    "num_leaves":       256,
    "max_depth":        -1,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "n_estimators":     2000,
    "verbose":          -1,
    "n_jobs":           -1,
    "random_state":     42,
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    
    lgb_oof[val_idx] = model.predict(X_val)
    lgb_test_preds  += model.predict(X_test) / N_SPLITS
    
    fold_score = max(0, 100 * r2_score(y_val, lgb_oof[val_idx]))
    print(f"  -> Fold {fold} Metric Score: {fold_score:.4f}")

lgb_cv_score = max(0, 100 * r2_score(y, lgb_oof))
print(f"==> Global LightGBM Local OOF Score: {lgb_cv_score:.4f} <===")

Training LightGBM Regressor across Folds...
  -> Fold 1 Metric Score: 99.2160
  -> Fold 2 Metric Score: 99.2304
  -> Fold 3 Metric Score: 99.2016
  -> Fold 4 Metric Score: 99.1505
  -> Fold 5 Metric Score: 99.2756
==> Global LightGBM Local OOF Score: 99.2158 <===


## 8. Model Training Round 2: XGBoost
XGBoost uses depth-wise growth to build structurally diverse trees compared to LightGBM's leaf-wise strategy. This difference makes them highly complementary components in an ensemble.

In [10]:
print("Training XGBoost Regressor across Folds...")

xgb_params = {
    "objective":         "reg:squarederror",
    "eval_metric":       "rmse",
    "learning_rate":     0.03,
    "max_depth":         7,
    "n_estimators":      2000,
    "subsample":         0.8,
    "colsample_bytree":  0.8,
    "reg_alpha":         0.1,
    "reg_lambda":        1.0,
    "min_child_weight":  5,
    "gamma":             0.0,
    "n_jobs":            -1,
    "random_state":      42,
    "verbosity":         0,
    "early_stopping_rounds": 100,
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    
    xgb_oof[val_idx] = model.predict(X_val)
    xgb_test_preds  += model.predict(X_test) / N_SPLITS
    
    fold_score = max(0, 100 * r2_score(y_val, xgb_oof[val_idx]))
    print(f"  -> Fold {fold} Metric Score: {fold_score:.4f}")

xgb_cv_score = max(0, 100 * r2_score(y, xgb_oof))
print(f"==> Global XGBoost Local OOF Score: {xgb_cv_score:.4f} <===")

Training XGBoost Regressor across Folds...
  -> Fold 1 Metric Score: 99.2705
  -> Fold 2 Metric Score: 99.2657
  -> Fold 3 Metric Score: 99.3037
  -> Fold 4 Metric Score: 99.2030
  -> Fold 5 Metric Score: 99.3181
==> Global XGBoost Local OOF Score: 99.2730 <===


## 9. Model Training Round 3: CatBoost
CatBoost uses symmetric trees, which helps guard against overfitting and provides robust out-of-the-box generalizations.

In [11]:
print("Training CatBoost Regressor across Folds...")

cat_params = dict(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3,
    subsample=0.8,
    colsample_bylevel=0.8,
    eval_metric="RMSE",
    early_stopping_rounds=100,
    random_seed=42,
    verbose=0,
)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = CatBoostRegressor(**cat_params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    
    cat_oof[val_idx] = model.predict(X_val)
    cat_test_preds  += model.predict(X_test) / N_SPLITS
    
    fold_score = max(0, 100 * r2_score(y_val, cat_oof[val_idx]))
    print(f"  -> Fold {fold} Metric Score: {fold_score:.4f}")

cat_cv_score = max(0, 100 * r2_score(y, cat_oof))
print(f"==> Global CatBoost Local OOF Score: {cat_cv_score:.4f} <===")

Training CatBoost Regressor across Folds...
  -> Fold 1 Metric Score: 99.3640
  -> Fold 2 Metric Score: 99.3475
  -> Fold 3 Metric Score: 99.3679
  -> Fold 4 Metric Score: 99.2576
  -> Fold 5 Metric Score: 99.3481
==> Global CatBoost Local OOF Score: 99.3380 <===


## 10. Meta-Ensemble Optimization (Nelder-Mead Solver)
Rather than guessing blending weights (like a static $33/33/33$ split), we formulate the blend as an optimization problem. We leverage SciPy's Nelder-Mead simplex algorithm to find the exact mathematical combination of our OOF arrays that maximizes our target evaluation metric ($R^2$).

In [12]:
print("Optimizing ensemble blend weights via Nelder-Mead...")

def objective_loss_func(w):
    """
    Minimizes negative R2 score to determine optimal weights.
    Includes clipping constraints to force weights between 0 and 1.
    """
    w = np.array(w)
    w = np.clip(w, 0, 1)
    if w.sum() == 0:
         return 0
    w = w / w.sum()
    
    # Linear combination blend
    blended_oof = w[0]*lgb_oof + w[1]*xgb_oof + w[2]*cat_oof
    return -r2_score(y, blended_oof)

# Begin search from uniform distributions
solver_res = minimize(
    objective_loss_func,
    x0=[1/3, 1/3, 1/3],
    method="Nelder-Mead",
    options={"maxiter": 5000, "xatol": 1e-8}
)

# Normalize solved weights to sum to 1.0
w_opt = np.clip(solver_res.x, 0, 1)
w_opt = w_opt / w_opt.sum()

print("\n--- Optimized Solution Found ---")
print(f"-> LightGBM Blend Component: {w_opt[0]:.4f}")
print(f"-> XGBoost Blend Component:  {w_opt[1]:.4f}")
print(f"-> CatBoost Blend Component: {w_opt[2]:.4f}")

# Map final training accuracy representation
oof_ensemble_predictions = w_opt[0]*lgb_oof + w_opt[1]*xgb_oof + w_opt[2]*cat_oof
final_ensemble_score = max(0, 100 * r2_score(y, oof_ensemble_predictions))

print(f"\n>>> Final Meta-Ensemble Local Cross-Validation Score: {final_ensemble_score:.4f} <<<")

Optimizing ensemble blend weights via Nelder-Mead...

--- Optimized Solution Found ---
-> LightGBM Blend Component: 0.0223
-> XGBoost Blend Component:  0.3767
-> CatBoost Blend Component: 0.6011

>>> Final Meta-Ensemble Local Cross-Validation Score: 99.3850 <<<


## 11. Inference Pipeline & Post-Processing
We now apply the optimal blending weights to generate our test predictions. We also apply a post-processing clip step to guarantee all outputs fall within valid physical boundaries for traffic demand ($[0.0, 1.0]$).

In [13]:
# Calculate final blended predictions
final_test_predictions = w_opt[0]*lgb_test_preds + w_opt[1]*xgb_test_preds + w_opt[2]*cat_test_preds

# Post-processing boundary clip
final_test_predictions = np.clip(final_test_predictions, 0.0, 1.0)

# Construct standard evaluation frame
submission = pd.DataFrame({
    "Index":  test["Index"].values,
    "demand": final_test_predictions,
})

# Save output array
submission.to_csv("submission.csv", index=False)
print("File exported successfully to -> submission.csv")

# Final verification snapshot
print("\nFirst 10 sample predictions for inspection:")
print(submission.head(10).to_string(index=False))

File exported successfully to -> submission.csv

First 10 sample predictions for inspection:
 Index   demand
     0 0.043274
     1 0.026679
     2 0.044390
     3 0.029235
     4 0.053926
     5 0.036889
     6 0.042893
     7 0.166390
     8 0.039057
     9 0.038291
